# Poster：从 recording 输入重新计算

在 hhw9l84 的 rfmapping 环境中从头运行。输入为保存的 RF counts/detection、HD classification、EBB frame counts/pose 和刺激记录；不读取以前的 poster 输出或统计缓存。

首个代码 cell 中的展示细胞、平滑参数及实验记录是明确配置；细胞数、比例、EBB 质量和示例排名都由当前输入计算。七鼠口径：全部 HD class 3；unassessed 在比例图计为 0 RF；不限制 ADN。原始 assessment 状态仍单独保留。


In [ ]:
from pathlib import Path
from hashlib import file_digest
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import load_npz

from Utils.direction_comparison import normalize_tc
from Utils.tc_comparison import calculate_tc_statistics
from scripts.rf_ebb_comparison import BAND_LABELS, pairwise_overview
from scripts.poster_rf_ebb import load_sorting_inputs, prepare_sorting_profiles
from EBB.quality import boundary_map_quality
from EBB.example import load_boundary_example, calculate_boundary_example
from EBB.spatial_cell_analysis import prepare_2d_projection
from HD.profiles import load_hd_tc, hd_profiles
from RF.free_moving import free_moving_exposure
from RF.paths import rf_source_path
from RF.rflocate.trials import _trial_records
from RF.profiles import prep_rf_tc, rf_centers
from RF.rflocate import load_rf, load_rfmap, rf_result_path
from Utils.plotting import LIGHT_PLOT_STYLE
from Utils.statistic_utils import smooth_tuning_curve
from scripts.export_m22_ebb_samples import mean_band_profiles, band_slices
from scripts.prepare_poster_candidates import sorting_source
from scripts.poster_population import POSTER_RECORDINGS
from scripts.slides_v9_figures import prepare_count_coverage, pool_count_coverage
from scripts.poster_figures import (centered_elevation, rf_rate_figure, rf_polar_figure,
    ebb_polar_figure, comparison_order, rotation_figure, coverage_figure)
from scripts.poster_design_figures import (PSTH_EDGES, rf_position_rates, response_moments,
    response_mean_sem, sample_overview_figure, rf_map_figure, timing_figure, fraction_figure,
    free_moving_figure, free_movement_overview_figure, ebb_sample_figure,
    poster_comparison_figure, hd_rf_figure, hd_rf_ebb_overview_figure, hd_ebb_joint_figure, hd_ebb_product_figure, annotated_region_arrows_figure)
from scripts.slides_v9_figures import region_positions_figure

%matplotlib inline
root = Path(os.environ.get('RFMAPPING_RECORDING_ROOT', '/mnt/senzailab/Kai/#Recording'))
base = root / 'm22/261008'
output_dir = Path(os.environ.get('POSTER_OUTPUT_DIR', 'poster_output/current'))
output_dir.mkdir(parents=True, exist_ok=True)
# User-selected illustrations; these IDs do not define population membership.
preview_unit = 269
multiple_rf_unit = 289
rotation_units = (287, 305)
example_units = sorted({preview_unit, multiple_rf_unit, *rotation_units})
# Match the saved EBB pipeline; verify the reconstructed rates below.
legacy_ebb_angular_sigma_deg, legacy_ebb_distance_sigma_bins = 30., 5.
ebb_example_spec = dict(mouse='m19', date='260831', session=2, probe='A', unit=125)
# User-preferred session; verify raw reconstruction and report temporal stability separately.
rf_ebb_sorting_spec = dict(mouse='m19', date='260827', probe='A', rf=2, hd=9, ebb=12)
# Exact saved analysis selected from the user's slides; missing files are errors.
slides_analysis_name = 'hd_ego_three_distance_polar_basler'
joint_hd_bins = 30
joint_smoothing_deg = 12.
arrow_hd_bins, arrow_hd_smoothing_deg = 180, 12.
arrow_annotation_unit = 307  # Illustrate HD and RF angles on this measured region.
# Native display geometry used by the legacy MATLAB free-moving generator.
fm_screen_width_px, fm_screen_deg = 960, 360
population_recordings = POSTER_RECORDINGS
population_anatomy = 'all_hd'
fraction_denominator = 'all_hd_missing_as_zero'
hd_rf_panel_aspect = 1.6  # Heatmap height / width; display only.
# Display-only exclusions requested for the sample; detector masks remain unchanged.
rf_highlight_exclusions = {269: [(4, 7), (4, 8), (4, 9)]}  # zero-based row, column
source_paths, figure_records = {}, []
plt.rcParams.update({**LIGHT_PLOT_STYLE, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'savefig.transparent': False, 'text.color': '#202020',
    'axes.labelcolor': '#202020', 'axes.edgecolor': '#202020', 'xtick.color': '#202020',
    'ytick.color': '#202020', 'grid.alpha': .5, 'font.size': 12,
    'font.family': 'DejaVu Sans', 'svg.fonttype': 'none'})
poster_style = {'font.size': 20, 'axes.titlesize': 22, 'axes.labelsize': 20,
                'xtick.labelsize': 18, 'ytick.labelsize': 18}

def source(name, path):
    source_paths[name] = Path(path)
    return Path(path)

def file_record(path):
    with Path(path).open('rb') as stream:
        digest = file_digest(stream, 'sha256').hexdigest()
    return dict(path=str(Path(path).resolve()), sha256=digest)

def save_figure(fig, stem, description, show=False, *, tight=True):
    for extension in ('svg', 'png'):
        fig.savefig(output_dir / f'{stem}.{extension}', dpi=200, bbox_inches='tight' if tight else None,
                    facecolor='white', transparent=False)
    figure_records.append(dict(stem=stem, description=description))
    if show:
        display(fig)
    plt.close(fig)

print('Configured recordings (mouse, date, HD session, RF session, probes):')
display(pd.DataFrame(population_recordings, columns=['mouse', 'date', 'HD_session', 'RF_session', 'probes']))


## 1. 同日 RF 与 HD：S2 / S9

选择保存的 excitatory ON RF+ 且 S9 HD class-3 的细胞。ON/OFF 都使用实际 presentations × 时间窗换算 Hz；绘图不修改原始 RF mask。


In [ ]:
hd_hz = {}
hd_saved_by_session = {}
hd_class3_ids = {}
for session in (9, 11):
    saved = load_hd_tc(source(f'HD_S{session}', base / f'261008_{session}/data/tuning_curves/ProbeA/tuning_curves.tc'))
    hd_saved_by_session[session] = saved
    table = hd_profiles(saved, probe='A', bins=30, smoothing_deg=0, hd_class=None)
    table.index = table.index.get_level_values('unit_id')
    hd_hz[session] = table
    hd_class3_ids[session] = np.asarray(saved['unit_id'])[np.asarray(saved['unit_data']['hd_class']) == 3]
    if session == 9:
        hd_ids = hd_class3_ids[session]
responses, masks, detected, rf_paths = {}, {}, {}, {}
for polarity, kind in [('ON', 'rfmapping'), ('OFF', 'rfmapping_off')]:
    path = source(f'RF_{polarity}', rf_source_path(base / '261008_2', 'A', root_kind=kind))
    rf_paths[polarity] = path
    result = load_rf(source(f'detection_{polarity}', rf_result_path(path, rf_type='excitatory')))
    detected[polarity] = result
    np.testing.assert_allclose(result.manifest['time_range_s'], [0, .2])
    maps = load_rfmap(path)
    presentations = maps[0].to_firing_rate(reconstruct_presentations=True).presentation_counts
    for unit in example_units:
        response = maps.by_unit_id(unit).to_firing_rate(presentation_counts=presentations).mean_rate(0, .2)
        responses[unit, polarity] = response
        masks[unit, polarity] = result.for_map(response)
        np.testing.assert_allclose(response.to_2d_array(),
            maps.by_unit_id(unit).sum(0, .2).to_2d_array() / (presentations * .2), rtol=1e-12, atol=1e-12)
    if polarity == 'ON':
        rf_ids = result.unit_ids[result.mask_2d.any(axis=(1, 2))]
        unit_ids = np.intersect1d(rf_ids, hd_ids)
        unit_keys = pd.Index([f'm22:261008:A:{unit}' for unit in unit_ids])
        native_rf = pd.DataFrame([maps.by_unit_id(int(unit)).sum(0, .2).sum_to_1d(axis='x', rf_only=False).to_1d_array()
                                  for unit in unit_ids], index=unit_keys, columns=maps[0].x_positions)
        native_rf.attrs.update(label='All-height RF', range=[-180, 180], response_units='spike_count', response_kind='RF')
        direct_projection = np.stack([maps.by_unit_id(int(unit)).sum(0, .2).to_2d_array().sum(axis=0)
                                      for unit in unit_ids])
        np.testing.assert_array_equal(native_rf.to_numpy(), direct_projection)
        rf_projection_max_abs_error = float(np.max(np.abs(native_rf.to_numpy() - direct_projection)))
    del maps
assert preview_unit in unit_ids
rf_profiles = prep_rf_tc(native_rf, bins=30, max_zero_bins=None)
hd_profiles_normalized = {}
for session in (9, 11):
    table = hd_hz[session].loc[unit_ids].copy()
    table.index = unit_keys
    hd_profiles_normalized[session] = table.div(table.max(axis=1), axis=0)
sort_sources = [sorting_source(base / f'261008_{session}', 'A') for session in (2, 7, 9, 10, 11)]
assert all(value == sort_sources[0] for value in sort_sources)
for session in (2, 7, 9, 10, 11):
    source(f'sorting_S{session}', base / f'261008_{session}/kilosort/ProbeA/kilosort_{session}/params.py')
for name in ('261008.mat', 'data/on_list_times.npy'):
    source(f'RF_presentations_{Path(name).name}', base / '261008_2' / name)
print('RF/HD:', len(unit_ids), f'cells; RF S2 / HD S9; ON/OFF unit {preview_unit}.')


In [ ]:
sample_masks = {polarity: masks[preview_unit, polarity].copy() for polarity in ('ON', 'OFF')}
for row, column in rf_highlight_exclusions.get(preview_unit, []):
    assert sample_masks['ON'][row, column]
    sample_masks['ON'][row, column] = False
for polarity in ('ON', 'OFF'):
    response, mask = responses[preview_unit, polarity], sample_masks[polarity]
    fig = rf_rate_figure(response, mask, preview_unit, polarity)
    save_figure(fig, f'02_rf_firing_rate_unit{preview_unit}_{polarity.lower()}', f'M22 S2 unit {preview_unit} {polarity}, Hz', tight=False)
    fig = rf_polar_figure(response, mask, preview_unit, polarity)
    save_figure(fig, f'01_rf_polar_unit{preview_unit}_{polarity.lower()}', f'M22 S2 unit {preview_unit} {polarity}, polar RF perimeter')
with plt.rc_context(poster_style):
    curve = smooth_tuning_curve(hd_hz[9].loc[preview_unit], sigma=1)
    fig = sample_overview_figure(curve, responses[preview_unit, 'ON'], responses[preview_unit, 'OFF'],
                                 sample_masks['ON'], sample_masks['OFF'], unit_id=preview_unit)
    save_figure(fig, '08_poster_hd_on_off_rf', f'S9 HD / S2 ON and OFF in rectangular and polar views, unit {preview_unit}', show=True, tight=False)

assert int(detected['ON'].for_unit(multiple_rf_unit, center_only=True).sum()) > 1
assert multiple_rf_unit in hd_ids
for polarity in ('ON', 'OFF'):
    fig = rf_rate_figure(responses[multiple_rf_unit, polarity], masks[multiple_rf_unit, polarity], multiple_rf_unit, polarity)
    save_figure(fig, f'02_rf_firing_rate_unit{multiple_rf_unit}_{polarity.lower()}',
                f'M22 S2 unit {multiple_rf_unit} {polarity}, Hz', tight=False)
    fig = rf_polar_figure(responses[multiple_rf_unit, polarity], masks[multiple_rf_unit, polarity], multiple_rf_unit, polarity)
    save_figure(fig, f'01_rf_polar_unit{multiple_rf_unit}_{polarity.lower()}',
                f'M22 S2 unit {multiple_rf_unit} {polarity}, polar RF perimeter')
with plt.rc_context(poster_style):
    curve = smooth_tuning_curve(hd_hz[9].loc[multiple_rf_unit], sigma=1)
    fig = sample_overview_figure(curve, responses[multiple_rf_unit, 'ON'], responses[multiple_rf_unit, 'OFF'],
        masks[multiple_rf_unit, 'ON'], masks[multiple_rf_unit, 'OFF'], unit_id=multiple_rf_unit)
    save_figure(fig, '17_poster_multiple_rf_sample', f'S9 HD / S2 ON and OFF in rectangular and polar views, multiple ON RF example {multiple_rf_unit}', show=True, tight=False)

# Standalone RF blocks follow the screenshot; native Illustrator HD plots can stay in place.
for unit in (preview_unit, multiple_rf_unit):
    outlines = sample_masks if unit == preview_unit else {polarity: masks[unit, polarity] for polarity in ('ON', 'OFF')}
    with plt.rc_context(poster_style):
        fig = rf_map_figure(responses[unit, 'ON'], responses[unit, 'OFF'],
            outlines['ON'], outlines['OFF'], unit_id=unit)
        save_figure(fig, f'20_poster_rf_map_unit{unit}',
            f'Unit {unit}: ON/OFF rectangles, one ON annulus and stepped outline', tight=False)

# Recompute the slides' M22 RF-region arrows from the current HD class-3 / ON-RF cohort.
arrow_hd = hd_profiles(hd_saved_by_session[9], probe='A', bins=arrow_hd_bins,
                        smoothing_deg=arrow_hd_smoothing_deg, hd_class=3)
arrow_hd.index = arrow_hd.index.get_level_values('unit_id')
arrow_peaks = arrow_hd.loc[unit_ids].idxmax(axis=1)
centers = rf_centers(detected['ON'], x_positions=detected['ON'].manifest['x_positions'],
                      y_positions=detected['ON'].manifest['y_positions'], probe='A')
region_pairs = centers.loc[pd.MultiIndex.from_product([['A'], unit_ids])].reset_index()
region_pairs['rf_region'] = region_pairs.groupby('unit_id', sort=False).cumcount() + 1
region_pairs['hd_peak_deg'] = region_pairs.unit_id.map(arrow_peaks)
region_pairs['direction_deg'] = (region_pairs.hd_peak_deg + region_pairs.rf_x_deg) % 360
region_pairs['unit_key'] = [f'm22:261008:A:{unit}' for unit in region_pairs.unit_id]
for unit, count in region_pairs.groupby('unit_id').size().items():
    assert count == detected['ON'].for_unit(int(unit), center_only=True).sum()
region_pairs.to_csv(output_dir / 'm22_poster_rf_region_arrows.csv', index=False)
region_arrow_statistics = dict(n_cells=int(region_pairs.unit_id.nunique()), n_regions=len(region_pairs),
    hd_bins=arrow_hd_bins, hd_smoothing_deg=arrow_hd_smoothing_deg, annotation_unit=arrow_annotation_unit,
    definition='Position: HD peak; arrow direction: (HD peak + saved ON RF region center X) modulo 360',
    selection='Current S9 HD class 3 and S2 ON RF; no ADN restriction',
    interpretation='One arrow per RF region; numerical angle sum is not a calibrated world-coordinate transform')
fig = annotated_region_arrows_figure(region_pairs, example_unit=arrow_annotation_unit)
save_figure(fig, '21_poster_m22_rf_region_arrows', 'M22 RF-region arrows, using the slides method and current HD cohort', show=True)


## 2. 旋转：287 / 305，S7 的两个 rotation 文件

三列依次为 S2 head-fixed、中间 `rotation_30_261008_7+.rfmap`、右侧 animal-centered `rotation_30_261008_7.rfmap`。分别使用文件内保存的刺激次数计算 Hz；色条从 0 到各图峰值，不更改 rotation offset。


In [ ]:
rotation = {(unit, 'head_fixed'): responses[unit, 'ON'] for unit in rotation_units}
rotation_ranges = []
paths = {
    'screen': base / '261008_7/data/rfmapping/good/-100_400_1ms/ProbeA/rotation_30/rotation_30_261008_7+.rfmap',
    'animal': base / '261008_7/data/rfmapping/good/-100_400_1ms/ProbeA/rotation_30/rotation_30_261008_7.rfmap',
}
for kind, path in paths.items():
    maps = load_rfmap(source(f'rotation_{kind}', path))
    presentations = maps[0].presentation_counts
    assert presentations is not None
    for unit in rotation_units:
        rotation[unit, kind] = maps.by_unit_id(unit).to_firing_rate(presentation_counts=presentations).mean_rate(0, .2)
        np.testing.assert_allclose(rotation[unit, kind].to_2d_array(),
            maps.by_unit_id(unit).sum(0, .2).to_2d_array() / (presentations * .2), rtol=1e-12, atol=1e-12)
    del maps
for (unit, kind), response in rotation.items():
    values = response.to_2d_array()
    rotation_ranges.append(dict(unit=unit, coordinates=kind, minimum_hz=float(np.nanmin(values)), maximum_hz=float(np.nanmax(values))))
fig = rotation_figure(rotation)
save_figure(fig, '07_poster_rotation_hz', f'M22 1008 units {rotation_units}: S2, S7 7+ middle, S7 rotation animal; Hz', show=True, tight=False)
display(pd.DataFrame(rotation_ranges))

rotation_alignment = []
for unit in rotation_units:
    reference_values = rotation[unit, 'head_fixed'].to_2d_array().ravel()
    for coordinates in ('screen', 'animal'):
        compared = rotation[unit, coordinates].to_2d_array().ravel()
        correlation = float(np.corrcoef(reference_values, compared)[0, 1])
        rotation_alignment.append(dict(unit=unit, coordinates=coordinates, correlation_with_head_fixed=correlation))
print('Descriptive rotation map correlations:', rotation_alignment)


## 3. M22 population and the selected M19 HD/EBB example

M22 comparisons retain S2 RF / S9 HD / S11 EBB. Native angular bins and recorded smoothing parameters are loaded; archived files without analysis_config explicitly use the historical 30° angular / 5-distance-bin smoothing, verified against saved rates. Counts and occupancy are reconstructed and checked.

The illustration is **M19 260831_2, Probe A, unit 125**, selected by the user. EBB and HD × EBB measured firing rates come from this same cell/session. Its split-half EBB stability is calculated below; selection does not imply RF positivity or independent conjunctive coding. A separate figure reproduces the reference's marginal-curve-product idea and is labeled dimensionless, not measured joint firing rate. Neither figure reuses fitted GLM components.

RF–EBB population comparisons use the session selected in `rf_ebb_sorting_spec`, independently of the example unit. Pairwise plots include **all saved distances, 0–8 cm, 8–16 cm, and >16 cm**, with the same RF reference, whole-cell permutations, and a shared cell-pair count color scale in the four-panel figure.


In [ ]:
wall_folder = base / '261008_11/data/spatial_cells/ProbeA/baseline'
wall_audit = json.loads(source('wall_audit', wall_folder / 'audit.json').read_text())
with np.load(source('wall_EBB_raw', wall_folder / 'restored_maps.npz')) as saved:
    rows = pd.Index(saved['unit_ids']).get_indexer(unit_ids)
    assert (rows >= 0).all()
    saved_ebb_rates = saved['egocentric_rate_maps'][rows].copy()
    saved_occupancy = saved['egocentric_occupancy']
    saved_spike_maps = saved['egocentric_spike_maps'][rows]
    theta_edges, distance_edges = saved['theta_edges'], saved['distance_edges']
    wall_frame_times, wall_frame_ids = saved['frame_times'], saved['frame_ids']
with np.load(source('wall_pose', wall_folder / 'position_correction.npz')) as pose:
    np.testing.assert_array_equal(pose['frame_ids'], wall_frame_ids)
    np.testing.assert_array_equal(pose['frame_times'], wall_frame_times)
    wall_projection = prepare_2d_projection(pose['theta_deg'][None, :], pose['theta_d_cm'],
                                            theta_edges, distance_edges)
    wall_held = pose['position_held']
    wall_hd_deg = pose['hd_deg']
wall_frame_counts = load_npz(source('wall_counts', wall_folder / 'frame_spike_counts.npz'))[rows]
ebb_smoothing_sigma = wall_audit.get('analysis_config', {}).get('smoothing_sigma_bins',
    [legacy_ebb_angular_sigma_deg / np.diff(theta_edges)[0], legacy_ebb_distance_sigma_bins])
wall_maps = boundary_map_quality(wall_projection, wall_frame_counts, wall_frame_times, wall_held,
                                 saved_occupancy.shape, sigma=ebb_smoothing_sigma)
occupancy, spike_maps = wall_maps['occupancy'], wall_maps['spike_maps']
ebb_display_rates, ebb_rates = wall_maps['display_rates'], wall_maps['rates']
np.testing.assert_allclose(occupancy, saved_occupancy, rtol=1e-12, atol=1e-9)
np.testing.assert_array_equal(spike_maps, saved_spike_maps)
np.testing.assert_allclose(ebb_display_rates, saved_ebb_rates, rtol=1e-12, atol=1e-12)
assert np.isnan(ebb_rates[:, occupancy == 0]).all()
wall_audit = json.loads(source('wall_audit', wall_folder / 'audit.json').read_text())
assert wall_audit['ebc_session'] == 11

ebb_candidates = pd.DataFrame(wall_maps['metrics'])
ebb_candidates.insert(0, 'unit', unit_ids)
ebb_candidates['s11_hd_class3'] = np.isin(unit_ids, hd_class3_ids[11])
ebb_candidates['on_rf'] = np.isin(unit_ids, rf_ids)
ebb_candidates['eligible'] = ebb_candidates.s11_hd_class3 & ebb_candidates.on_rf
ebb_candidates = ebb_candidates.sort_values(['descriptive_score', 'unit'], ascending=[False, True])
ebb_candidates.to_csv(output_dir / 'ebb_example_candidates.csv', index=False)
eligible_ebb = ebb_candidates.loc[ebb_candidates.eligible].dropna(subset=['descriptive_score'])
if eligible_ebb.empty:
    raise ValueError('No paired S11 HD-class-3 / ON-RF cell has a defined EBB split-half score')
ebb_preview_unit = int(eligible_ebb.iloc[0]['unit'])
ebb_example_quality = eligible_ebb.iloc[0].to_dict()
quality = dict(**wall_maps['summary'],
    smoothing_sigma_bins=ebb_smoothing_sigma,
    angular_sigma_deg=float(ebb_smoothing_sigma[0] * np.diff(theta_edges)[0]),
    distance_sigma_cm=float(ebb_smoothing_sigma[1] * np.diff(distance_edges)[0]),
    arena_size_cm=wall_audit['arena_size_cm'], raw_reconstruction_passed=True,
    occupancy_max_abs_error_s=float(np.max(np.abs(occupancy - saved_occupancy))),
    spike_map_max_abs_error=float(np.max(np.abs(spike_maps - saved_spike_maps))),
    rate_map_max_abs_error_hz=float(np.nanmax(np.abs(ebb_display_rates - saved_ebb_rates))),
    selected_unit=ebb_preview_unit,
    selection_rule='S11 HD class 3 within paired ON RF cohort; max(half_r, 0) * (q95-q05)/q95; unit ID breaks ties',
    units=json.loads(ebb_candidates.to_json(orient='records', double_precision=15)))
(output_dir / 'm22_261008_wall_quality.json').write_text(json.dumps(quality, indent=2, allow_nan=False) + '\n')
np.savez_compressed(output_dir / 'ebb_quality_maps.npz', unit_ids=unit_ids,
    rates=ebb_rates, half_rates=wall_maps['half_rates'], unheld_rates=wall_maps['unheld_rates'],
    occupancy_s=occupancy, half_occupancy_s=wall_maps['half_occupancy'],
    unheld_occupancy_s=wall_maps['unheld_occupancy'], theta_edges=theta_edges, distance_edges=distance_edges)
print(f"Computed EBB example: unit {ebb_preview_unit}; split-half r={ebb_example_quality['half_r']:.3f}")
display(eligible_ebb[['unit', 'half_r', 'modulation_q95_q05', 'descriptive_score', 'mean_hz']])

profiles_rf = {'RF': rf_profiles, 'HD': hd_profiles_normalized[9]}
profiles_wall = {'HD': hd_profiles_normalized[11]}
comparison_theta_edges = theta_edges[::(len(theta_edges) - 1) // 30]
angles = (comparison_theta_edges[:-1] + comparison_theta_edges[1:]) / 2
for band, values in mean_band_profiles(ebb_rates, distance_edges).items():
    table = pd.DataFrame(values, index=unit_keys, columns=angles)
    table.to_csv(output_dir / f'ebb_{band}_hz.csv')
    normalized = table.div(table.max(axis=1), axis=0)
    profiles_rf[band] = profiles_wall[band] = normalized
orders = {'RF': comparison_order(rf_profiles), 'HD9': comparison_order(profiles_rf['HD']),
          'HD11': comparison_order(profiles_wall['HD'])}
for name, order in orders.items():
    pd.DataFrame({'rank': np.arange(1, len(order) + 1), 'unit_key': order}).to_csv(output_dir / f'{name}_order.csv', index=False)
for prefix, profiles in [('rf_hd9', profiles_rf), ('ebb_hd11', profiles_wall)]:
    for key, table in profiles.items():
        assert table.index.equals(unit_keys) and np.isfinite(table.to_numpy()).all()
        table.to_csv(output_dir / f'{prefix}_{key}_profiles.csv')
# The user-selected illustration is independent of the M22 population above.
example_loaded = load_boundary_example(root, ebb_example_spec)
for name, path in example_loaded['source_paths'].items():
    source('example_' + name, path)
example = calculate_boundary_example(example_loaded, hd_bins=joint_hd_bins,
    joint_smoothing_deg=joint_smoothing_deg, legacy_angular_sigma_deg=legacy_ebb_angular_sigma_deg,
    legacy_distance_sigma_bins=legacy_ebb_distance_sigma_bins)
ebb_preview_unit = ebb_example_spec['unit']
ebb_example_quality = example['quality']['metrics'][0]
example_label = f"{ebb_example_spec['mouse'].upper()} {ebb_example_spec['date']}_{ebb_example_spec['session']} · Probe {ebb_example_spec['probe']}"
joint_maps = example['joint']
joint_payload = dict(unit_id=ebb_preview_unit, hd_edges=joint_maps['hd_edges'], theta_edges=example['theta_edges'],
    distance_edges=example['distance_edges'], spike_maps=joint_maps['spike_maps'], occupancy_maps=joint_maps['occupancy_maps'])
joint_rows = []
hd_centers = (joint_maps['hd_edges'][:-1] + joint_maps['hd_edges'][1:]) / 2
bearing_centers = (example['theta_edges'][:-1] + example['theta_edges'][1:]) / 2
for band, values in joint_maps['bands'].items():
    joint_payload.update({f'{band}_{name}': value for name, value in values.items()})
    joint_payload[f'{band}_marginal_product'] = example['products'][band]
    hi, bi = np.indices(values['hz'].shape)
    joint_rows.append(pd.DataFrame(dict(band=band, hd_deg=hd_centers[hi.ravel()],
        bearing_deg=bearing_centers[bi.ravel()], spike_counts=values['spike_counts'].ravel(),
        occupancy_s=values['occupancy_s'].ravel(), raw_hz=values['raw_hz'].ravel(), display_hz=values['hz'].ravel())))
np.savez_compressed(output_dir / f'hd_ebb_joint_unit{ebb_preview_unit}.npz', **joint_payload)
pd.concat(joint_rows, ignore_index=True).to_csv(output_dir / f'hd_ebb_joint_unit{ebb_preview_unit}.csv', index=False)
joint_metadata = dict(**ebb_example_spec, hd_bins=joint_hd_bins,
    bearing_bins=len(example['theta_edges'])-1, smoothing_deg=joint_smoothing_deg,
    hd_source='position_correction.npz: hd_deg (native clockwise HD)',
    display_hd_ticks_top_to_bottom=[0,90,180,270,0],
    bearing_source='saved theta_deg and theta_d_cm; clockwise from calibrated heading',
    distance_range_cm=example['distance_edges'][[0,-1]].tolist(),
    rate_method='frame spikes / frame occupancy conditional on HD and boundary bearing; separate angular smoothing',
    product_method='normalized same-session saved HD curve times normalized marginal EBB curve; dimensionless',
    raw_marginals_match_saved=True, n_frames=joint_maps['n_frames'],
    selection='User-selected M19 260831_2 unit 125; HD class 3; not selected for RF positivity',
    ebb_quality=ebb_example_quality)
(output_dir / f'hd_ebb_joint_unit{ebb_preview_unit}.json').write_text(json.dumps(joint_metadata, indent=2) + '\n')
print(example_label, 'unit', ebb_preview_unit, 'EBB split-half r =', ebb_example_quality['half_r'])
# Preserve the measured joint result as a separate export; the poster uses the user's slides product.
with plt.rc_context(poster_style):
    fig = hd_ebb_joint_figure(joint_maps, unit_id=ebb_preview_unit, source_label=example_label)
    save_figure(fig, f'24_measured_hd_ebb_joint_unit{ebb_preview_unit}',
        example_label + ': measured conditional HD × EBB Hz; separate from the poster product')
slides_folder = (root / ebb_example_spec['mouse'] / ebb_example_spec['date'] /
    f"{ebb_example_spec['date']}_{ebb_example_spec['session']}" / 'data/glm' / slides_analysis_name /
    f"Probe{ebb_example_spec['probe']}" / 'baseline' / f"unit_{ebb_preview_unit}")
slides_settings = json.loads(source('slides_product_settings', slides_folder / 'settings.json').read_text())
assert slides_settings['config']['unit_id'] == ebb_preview_unit
with np.load(source('slides_product_arrays', slides_folder / 'arrays.npz'), allow_pickle=False) as saved:
    slides_recorded = saved['component_map__Recorded']
    slides_glm = saved['component_map__GLM']
    slides_hd_angles = saved['tuning_hd_angles_deg']
    slides_ego_angles = saved['tuning_ego_angles_deg']
    slides_glm_angles = saved['angles_deg']
    slides_hd_hz = saved['tuning_recorded_hd_rate_hz']
    slides_hd_occupancy = saved['tuning_recorded_hd_occupancy_s']
    slides_ego_hz = saved['tuning_recorded_ego_rate_hz']
# Verify the stored product from its own saved marginal curves, without refitting or substituting curves.
slides_hd_masked = np.where(slides_hd_occupancy >= slides_settings['min_occupancy_s'], slides_hd_hz, np.nan)
slides_expected = ((slides_hd_masked / np.nanmax(slides_hd_masked, axis=1, keepdims=True))[:, :, None]
    * (slides_ego_hz / np.nanmax(slides_ego_hz, axis=1, keepdims=True))[:, None, :])
np.testing.assert_allclose(slides_recorded, slides_expected, rtol=0, atol=0)
# The saved GLM analysis uses native CCW angles; convert coordinates and carry the values with them.
slides_hd_cw, slides_ego_cw = (-slides_hd_angles) % 360, (-slides_ego_angles) % 360
slides_hd_order, slides_ego_order = np.argsort(slides_hd_cw), np.argsort(slides_ego_cw)
slides_hd_edges = np.linspace(0, 360, len(slides_hd_angles) + 1)
slides_theta_edges = np.linspace(0, 360, len(slides_ego_angles) + 1)
np.testing.assert_allclose(slides_hd_cw[slides_hd_order], (slides_hd_edges[:-1] + slides_hd_edges[1:]) / 2)
np.testing.assert_allclose(slides_ego_cw[slides_ego_order], (slides_theta_edges[:-1] + slides_theta_edges[1:]) / 2)
slides_products = {band: slides_recorded[index][np.ix_(slides_hd_order, slides_ego_order)]
    for index, band in enumerate(('0-8', '8-16', '16-inf'))}
slides_example = dict(products=slides_products, joint=dict(hd_edges=slides_hd_edges), theta_edges=slides_theta_edges)
slides_product_metadata = dict(source=str(slides_folder), unit=ebb_preview_unit,
    calculation=slides_settings['recorded_vs_three_glms']['calculation'],
    units='Dimensionless product of normalized marginal tunings; not joint firing rate',
    display_hd_ticks_top_to_bottom=[0,90,180,270,0], saved_product_verified_exact=True,
    poster_panels='The three Recorded curves panels from the selected slides',
    native_angle_convention='CCW in the saved analysis; converted to CW by negating coordinates and reordering values',
    source_version=slides_analysis_name)
(output_dir / 'slides_hd_ebb_product_metadata.json').write_text(json.dumps(slides_product_metadata, indent=2) + '\n')
np.savez_compressed(output_dir / 'slides_hd_ebb_product_arrays.npz', recorded=slides_recorded, glm=slides_glm,
    hd_angles_native=slides_hd_angles, ego_angles_native=slides_ego_angles, glm_angles_native=slides_glm_angles,
    hd_hz=slides_hd_hz, hd_occupancy_s=slides_hd_occupancy, ebb_hz=slides_ego_hz)
with plt.rc_context(poster_style):
    fig = hd_ebb_product_figure(slides_example, unit_id=ebb_preview_unit,
        source_label=example_label + ' · saved slides analysis')
    save_figure(fig, f'19_poster_hd_ebb_product_unit{ebb_preview_unit}',
        example_label + ': exact saved Recorded marginal-curve product from the selected slides', show=True)
    fig = hd_ebb_product_figure(example, unit_id=ebb_preview_unit, source_label=example_label)
    save_figure(fig, f'22_hd_ebb_marginal_product_unit{ebb_preview_unit}',
        example_label + ': current spatial/HD marginal product; separate from saved slides')
for guides in (False, True):
    fig = ebb_polar_figure(example['rate'], example['theta_edges'], example['distance_edges'],
                          ebb_preview_unit, guides=guides, mouse=example_label)
    stem = f'04_ebb_polar_8_16cm_unit{ebb_preview_unit}' if guides else f'03_ebb_polar_unit{ebb_preview_unit}'
    save_figure(fig, stem, example_label + f' unit {ebb_preview_unit}: EBB Hz')
with plt.rc_context(poster_style):
    fig = ebb_sample_figure(example['rate'], example['theta_edges'], example['distance_edges'],
                           unit_id=ebb_preview_unit, source_label=example_label)
    save_figure(fig, '12_poster_ebb_polar_rectangular', example_label + f' unit {ebb_preview_unit}: EBB Hz', show=True)
del wall_projection, wall_frame_counts, example_loaded
# Load the selected population directly from recordings, independently of the example cell.
sorting_loaded = load_sorting_inputs(root, rf_ebb_sorting_spec)
for name, path in sorting_loaded['source_paths'].items():
    source('rf_ebb_sorting_' + name, path)
sorting = prepare_sorting_profiles(sorting_loaded,
    legacy_angular_sigma_deg=legacy_ebb_angular_sigma_deg,
    legacy_distance_sigma_bins=legacy_ebb_distance_sigma_bins)
sorting_order = comparison_order(sorting['profiles']['RF'])
sorting['metadata']['order'] = sorting_order.tolist()
sorting['metadata']['session_selection'] = (
    f"User-selected {rf_ebb_sorting_spec['mouse'].upper()} {rf_ebb_sorting_spec['date']}, "
    f"RF S{rf_ebb_sorting_spec['rf']} / EBB S{rf_ebb_sorting_spec['ebb']} / HD S{rf_ebb_sorting_spec['hd']}; all paired cells retained. "
    'Full spatial coverage does not imply temporal stability: inspect exported half-map correlations.')
sorting_label = (f"{rf_ebb_sorting_spec['mouse'].upper()} {rf_ebb_sorting_spec['date']} · "
    f"RF S{rf_ebb_sorting_spec['rf']} / EBB S{rf_ebb_sorting_spec['ebb']} · "
    f"{len(sorting_order)} HD cells · sorted by RF peak")
for name, table in sorting['profiles'].items():
    table.to_csv(output_dir / f'rf_ebb_sorting_{name}_source_profiles.csv')
    normalize_tc(table).loc[sorting_order].to_csv(output_dir / f'rf_ebb_sorting_{name}_normalized.csv')
sorting['native_rf'].to_csv(output_dir / 'rf_ebb_sorting_native_rf_counts.csv')
sorting['metrics'].to_csv(output_dir / 'rf_ebb_sorting_unit_quality.csv')
(output_dir / 'rf_ebb_sorting_metadata.json').write_text(json.dumps(sorting['metadata'], indent=2) + '\n')
display(sorting['metrics'][['half_r', 'unheld_r', 'mean_hz']])
del sorting_loaded
# Same cell-pair calculation for all distances and each of the three distance bands.
poster_pairwise, pairwise_metadata = {}, {}
rf_pairwise_profiles = normalize_tc(sorting['profiles']['RF'])
pairwise_title = f"RF–EBB pairwise · {rf_ebb_sorting_spec['mouse'].upper()} {rf_ebb_sorting_spec['date']}"
for band, band_label in BAND_LABELS.items():
    stats = calculate_tc_statistics(rf_pairwise_profiles,
        normalize_tc(sorting['profiles'][band]), bins=15, is_wrap=True,
        n_permutations=10000, random_seed=1)
    poster_pairwise[band] = stats
    stats['peaks'].to_csv(output_dir / f'rf_ebb_pairwise_{band}_peak_angles.csv')
    pairwise_metadata[band] = dict(cells=len(stats['peaks']),
        unique_pairs=len(stats['peaks']) * (len(stats['peaks']) - 1) // 2,
        distance_band=band_label.lower(), statistics=stats['direction']['pairwise'],
        n_permutations=10000, random_seed=1,
        interpretation='Descriptive same-sorting cohort; temporal stability is reported separately.')
    (output_dir / f'rf_ebb_pairwise_{band}_statistics.json').write_text(
        json.dumps(pairwise_metadata[band], indent=2) + '\n')
    fig = pairwise_overview({band: stats}, title=pairwise_title)
    save_figure(fig, f'25_poster_rf_ebb_pairwise_{band}',
        f"RF S{rf_ebb_sorting_spec['rf']} / EBB S{rf_ebb_sorting_spec['ebb']}: "
        f'unique between-cell peak-distance pairs; {band_label}')
fig = pairwise_overview(poster_pairwise, title=pairwise_title)
save_figure(fig, '26_poster_rf_ebb_pairwise_four_bands',
    f"RF S{rf_ebb_sorting_spec['rf']} / EBB S{rf_ebb_sorting_spec['ebb']}: "
    'all distances, 0–8 cm, 8–16 cm, >16 cm; shared cell-pair count color scale')
fig = pairwise_overview(poster_pairwise, title=pairwise_title, single_row=True)
save_figure(fig, '27_poster_rf_ebb_pairwise_row',
    'Same four RF–EBB pairwise comparisons in one row for the current poster layout', show=True)

with plt.rc_context(poster_style):
    fig = poster_comparison_figure({key: normalize_tc(table) for key, table in sorting['profiles'].items()},
        sorting_order, 'RF', source_label=sorting_label)
    save_figure(fig, '13_poster_rf_ebb_sorted', sorting_label + '; each cell normalized from 0 to max', show=True)
    fig = hd_rf_figure(profiles_rf, orders['HD9'], panel_aspect=hd_rf_panel_aspect)
    save_figure(fig, '14_poster_hd_rf_sorted', f'S9 HD / S2 RF, {len(unit_ids)} cells sorted by S9 HD', show=True)
    fig = poster_comparison_figure(profiles_wall, orders['HD11'], 'HD', include_reference=False)
    save_figure(fig, '15_poster_hd_ebb_three_bands', f'S11 EBB, {len(unit_ids)} cells ordered by same-session S11 HD', show=True)
    fig = poster_comparison_figure(profiles_wall, orders['HD11'], 'HD')
    save_figure(fig, '06_hd_vs_ebb_three_distances_sorted_by_hd', f'Full S11 HD / S11 EBB reference, all {len(unit_ids)} cells')

with plt.rc_context(poster_style):
    fig = hd_rf_ebb_overview_figure(profiles_rf, orders['HD9'], profiles_wall, orders['HD11'])
    save_figure(fig, '16_poster_hd_rf_ebb_shared_scale', 'HD/RF and same-session HD/EBB, separate preserved row orders and one shared normalized colorbar', show=True, tight=False)


## 4. 自由活动视觉 RF：S10 条形刺激

配置的展示细胞的实际 0–200 ms firing rate。每次从原始刺激 MAT、onset、已校正 XY/HD 和 calibration 重算每个 bin 的 presentation count，并与保存的 occupancy 核对。这里只有一个 elevation 维度，因此用完整高度条纹呈现，不把它复制成多行 RF。中栏横排视觉 RF、EBB polar、EBB rectangle。视觉 RF 与 head-fixed 图均为 8 英寸宽、84:360 高宽比；这只是显示比例，不添加 elevation 采样。


In [ ]:
fm_path = source('FM_RF', base / '261008_10/data/rfmapping/good/-100_400_1ms/ProbeA/free_moving/free_moving_261008_10.rfmap')
fm_maps = load_rfmap(fm_path)
fm_session = base / '261008_10'
fm_trials = _trial_records(source('FM_trials', fm_session / '261008.mat'))
fm_onsets = np.load(source('FM_onsets', fm_session / 'data/on_list_times.npy')).squeeze()
fm_xy = np.load(source('FM_xy', fm_session / 'data/position_trials_time.npy'))
fm_hd_px = np.load(source('FM_hd', fm_session / 'data/hd_trials_times.npy')).squeeze()
fm_calibration = json.loads(source('FM_calibration', fm_session / '261008.calib').read_text())
fm_exposure = free_moving_exposure(fm_trials, fm_onsets, fm_xy, fm_hd_px, fm_calibration,
    n_x_bins=len(fm_maps[0].x_positions), screen_width_px=fm_screen_width_px, screen_deg=fm_screen_deg)
fm_presentations = fm_exposure['presentation_counts']
fm_saved_occupancy = np.asarray(fm_maps[0].metadata['occupancyTimeSec']).ravel()
np.testing.assert_allclose(fm_exposure['occupancy_s'], fm_saved_occupancy, atol=1e-9, rtol=1e-10)
fm_occupancy_max_abs_error = float(np.max(np.abs(fm_exposure['occupancy_s'] - fm_saved_occupancy)))
np.savez_compressed(output_dir / 'free_moving_exposure.npz', presentation_counts=fm_presentations,
    occupancy_reconstructed=fm_exposure['occupancy_s'], occupancy_saved=fm_saved_occupancy)
fm_unit = fm_maps.by_unit_id(preview_unit)
assert len(fm_unit.y_positions) == 1
fm_hz = fm_unit.sum(0, .2).to_2d_array()[0] / (fm_presentations * .2)
assert np.isfinite(fm_hz).all() and (fm_presentations > 0).all()
paired_sample_row = int(np.flatnonzero(unit_ids == preview_unit)[0])
with plt.rc_context(poster_style):
    fig = free_moving_figure(fm_unit.x_positions, fm_hz)
    save_figure(fig, f'11_free_moving_bar_rf_unit{preview_unit}', f'M22 S10 unit {preview_unit}, full-height bar RF, 0–200 ms Hz', tight=False)
    fig = free_movement_overview_figure(fm_unit.x_positions, fm_hz, ebb_display_rates[paired_sample_row], theta_edges, distance_edges, unit_id=preview_unit)
    save_figure(fig, '11_poster_free_movement_row', f'Same unit {preview_unit}: S10 bar RF, S11 EBB polar and rectangle', show=True, tight=False)
del fm_maps


## 5. 七鼠覆盖、比例、响应时间和方向

按用户确认：全部 HD class-3 作比例分母，unassessed 计入 0 RF；不使用 ADN 限制。各主图使用同一份当前 HD class-3 与 excitatory ON RF 阳性交集，完整 mouse/date/probe/unit key 防止跨探针重号。原始 assessment 状态保留在 CSV，绘图口径单列。原始 RF QC、GUI smoothing、检测 mask/center 和共同 sorting 来源逐鼠重新验证，但不修改源数据。PSTH 使用完整时间轴，合并所有细胞 RF mask 内的空间 bins，每个 bin 等权计算 mean 和描述性 SEM；同一细胞的 bins 不视为独立生物学重复。逐细胞曲线及基于细胞均值的等权动物平均另存。M14 最外两行只在 coverage 中截去。


In [ ]:
from RF.rflocate import RFMapList, detect_rf
from RF.rflocate.workflow import rf_bin_qc
from RF.rflocate.smoothing import smooth_rf_maps
from Utils.direction_comparison import concat, normalize_tc
from scripts.poster_population import (RF_CATEGORIES,
    population_unit_table, select_hd_population, fraction_summary)
from scripts.slides_v9_figures import peak_sum_data, peak_sum_figure, seven_mice_figure

# One membership table controls coverage, timing, fraction and direction panels.
coverage_panels, coordinate_records, timing_records, population_tables = [], [], [], []
verification_records, cell_time_rows, direction_tables = [], [], {}
pooled_position_rates = []
all_rf_count = 0
for mouse, date, hd_session, rf_session, probes in population_recordings:
    mouse_hd, mouse_rf = [], []
    for probe in probes:
        recording = root / mouse / date
        rf_folder, hd_folder = (recording / f'{date}_{session}' for session in (rf_session, hd_session))
        path = source(f'coverage_{mouse}_{probe}', rf_source_path(rf_folder, probe))
        result = load_rf(source(f'coverage_detection_{mouse}_{probe}', rf_result_path(path)))
        hd_file = source(f'fraction_HD_{mouse}_{probe}', hd_folder / f'data/tuning_curves/Probe{probe}/tuning_curves.tc')
        saved_hd = load_hd_tc(hd_file)
        for name, folder, session in [('RF', rf_folder, rf_session), ('HD', hd_folder, hd_session)]:
            source(f'population_sorting_{mouse}_{probe}_{name}',
                   folder / f'kilosort/Probe{probe}/kilosort_{session}/params.py')
        assert sorting_source(rf_folder, probe) == sorting_source(hd_folder, probe)
        maps = load_rfmap(path)
        presentations = maps[0].to_firing_rate(reconstruct_presentations=True).presentation_counts
        mean_rates = RFMapList([item.to_firing_rate(presentation_counts=presentations).mean_rate(0, .2)
                               for item in maps], path)
        qc = rf_bin_qc(mean_rates, max_zero_bins=2)
        np.testing.assert_array_equal(qc['unit_ids'][qc['keep']], result.unit_ids)
        selected_rates = RFMapList([item for item, keep in zip(mean_rates, qc['keep']) if keep], path)
        parameters = result.manifest['parameters']
        assert parameters['alternative'] == 'greater'
        np.testing.assert_allclose(result.manifest['time_range_s'], [0, .2], atol=1e-12)
        recomputed = detect_rf(smooth_rf_maps(selected_rates), reference=parameters['reference'],
            sigma=parameters['cluster_forming_z'], drop_bins=parameters['drop_bins'],
            surround_decay=parameters['surround_decay'], is_shuffle=False,
            exclude_zero_bins=parameters['exclude_zero_bins'], wrap_x=parameters['wrap_x'], show_progress=False)
        np.testing.assert_array_equal(recomputed.mask_2d, result.mask_2d)
        np.testing.assert_array_equal(recomputed.center_2d, result.center_2d)
        verification_records.append(dict(mouse=mouse, date=date, probe=probe,
            raw_units=len(maps), assessed_units=len(result.unit_ids),
            raw_qc_matches_saved=True, recomputed_masks_match=True, recomputed_centers_match=True,
            sorting_sources_match=True, detector=result.manifest['detector_algorithm'], parameters=parameters))

        adn_path = rf_folder / 'data/adn_neuron.npy'
        adn_ids = None
        if adn_path.exists():
            adn = np.load(source(f'population_ADN_{mouse}_{probe}', adn_path), allow_pickle=False)
            adn_ids = adn['unit_id'][adn['probe'] == f'Probe{probe}']
        table = population_unit_table(saved_hd, result, maps.unit_ids, qc, adn_ids=adn_ids)
        table['mouse'], table['date'], table['probe'] = mouse, date, probe
        table['hd_session'], table['rf_session'] = hd_session, rf_session
        hd_population = select_hd_population(table, anatomy=population_anatomy)
        ids = hd_population.index.to_numpy()
        positive_ids = hd_population.index[hd_population.n_on_rf.gt(0).fillna(False)].to_numpy()
        table['in_poster_hd_population'] = table.index.isin(ids)
        table['in_poster_rf_positive_population'] = table.index.isin(positive_ids)
        population_tables.append(table.reset_index())
        all_rf_count += int(result.mask_2d.any(axis=(1, 2)).sum())

        native_x, native_y = np.asarray(result.manifest['x_positions']), np.asarray(result.manifest['y_positions'])
        measured = np.asarray(maps[0].metadata['occupancyTimeSec']).reshape(len(native_y), len(native_x)) > 0
        keep = abs(centered_elevation(native_y)) <= 36
        panel = prepare_count_coverage(result, measured, rows=keep, unit_ids=ids)
        origin = native_y[len(native_y) // 2]
        panel['y'], panel['y_edges'] = panel['y'] - origin, panel['y_edges'] - origin
        native_masks = result.mask_2d[np.isin(result.unit_ids, ids)][:, keep] & measured[None, keep]
        np.testing.assert_array_equal(panel['raw_count'], native_masks.sum(axis=0))
        np.testing.assert_array_equal(panel['azimuth_count'], native_masks.any(axis=1).sum(axis=0))
        np.testing.assert_array_equal(panel['elevation_count'], native_masks.any(axis=2).sum(axis=0))
        assert panel['n_rf'] == len(positive_ids)
        coverage_panels.append(panel)
        np.savez_compressed(output_dir / f'seven_mice_coverage_{mouse}_{probe}.npz',
                            unit_ids=ids, positive_unit_ids=positive_ids, **panel)
        coordinate_records.append(dict(mouse=mouse, probe=probe, source_y=native_y.tolist(),
            native_origin_deg=float(origin), kept_rows=np.flatnonzero(keep).tolist(),
            dropped_rows=np.flatnonzero(~keep).tolist(), centered_elevation=centered_elevation(native_y)[keep].tolist(),
            n_rf_before=int(result.mask_2d[np.isin(result.unit_ids, ids)].any(axis=(1, 2)).sum()), n_rf_after=panel['n_rf']))

        positions = 0
        for uid in positive_ids:
            mask = result.for_unit(int(uid)).astype(bool)
            rates = maps.by_unit_id(int(uid)).to_firing_rate(presentation_counts=presentations)
            position_rates = rf_position_rates(rates, mask)
            pooled_position_rates.append(position_rates)
            mean = np.nanmean(position_rates, axis=0)
            if mouse == 'm22' and uid == preview_unit:
                sample_moments = response_moments(position_rates)
            for time_ms, value in zip((PSTH_EDGES[:-1] + PSTH_EDGES[1:]) * 500, mean):
                cell_time_rows.append(dict(mouse=mouse, date=date, probe=probe, unit_id=int(uid),
                                          time_ms=time_ms, mean_hz=float(value), n_rf_positions=int(mask.sum())))
            positions += int(mask.sum())
        timing_records.append(dict(mouse=mouse, date=date, session=rf_session, probe=probe,
                                   n_units=len(positive_ids), n_positions=positions))

        keys = pd.Index([f'{mouse}:{date}:{probe}:{uid}' for uid in positive_ids], name='unit_id')
        hd_table = hd_profiles(saved_hd, probe=probe, bins=30, smoothing_deg=0, hd_class=3)
        hd_table.index = hd_table.index.get_level_values('unit_id')
        hd_table = normalize_tc(hd_table.loc[positive_ids])
        hd_table.index = keys
        hd_table.attrs = dict(label='HD', range=[-180, 180], response_units='normalized', response_kind='HD')
        raw = pd.DataFrame([maps.by_unit_id(int(uid)).sum(0, .2).sum_to_1d(axis='x', rf_only=False).to_1d_array()
                            for uid in positive_ids], index=keys, columns=maps[0].x_positions)
        raw.attrs.update(label='All-height RF', range=[-150, 150] if mouse == 'm14' else [-180, 180],
                         response_units='spike_count', response_kind='RF')
        mouse_hd.append(hd_table)
        mouse_rf.append(prep_rf_tc(raw, bins=30, max_zero_bins=None))
        del maps, mean_rates, selected_rates
    direction_tables[mouse] = peak_sum_data(concat(*mouse_hd, label=f'{mouse} HD'),
                                            concat(*mouse_rf, label=f'{mouse} RF'), bins=15)

population_units = pd.concat(population_tables, ignore_index=True)
assert not population_units.duplicated(['mouse', 'date', 'probe', 'unit_id']).any()
fraction_units = population_units.loc[population_units.in_poster_hd_population].copy()
assessment_counts = fraction_units.category.value_counts().reindex(RF_CATEGORIES, fill_value=0)
fraction_counts = assessment_counts.drop('RF unassessed').copy()
fraction_counts['0 ON RF'] += assessment_counts['RF unassessed']
fraction_units['n_on_rf_for_ratio'] = fraction_units.n_on_rf.fillna(0)
fraction_units['ratio_category'] = fraction_units.category.replace({'RF unassessed': '0 ON RF'})
population_summary = fraction_summary(fraction_units)
population_summary['n_zero_for_ratio'] = int(fraction_counts['0 ON RF'])
population_summary['on_rf_percent'] = population_summary['on_rf_percent_all_hd']
anatomy_summaries = {name: fraction_summary(select_hd_population(population_units, anatomy=name))
                    for name in ('all_hd', 'm23_adn', 'available_adn')}
population_units.to_csv(output_dir / 'seven_mice_population_units.csv', index=False)
fraction_units.to_csv(output_dir / 'seven_mice_on_rf_fraction_units.csv', index=False)
fraction_counts.to_csv(output_dir / 'seven_mice_on_rf_fraction_counts.csv', header=['n_cells'])
assessment_counts.to_csv(output_dir / 'seven_mice_on_rf_assessment_counts.csv', header=['n_cells'])
fraction_by_recording = pd.DataFrame([dict(mouse=mouse, date=date, probe=probe, **fraction_summary(group))
    for (mouse, date, probe), group in fraction_units.groupby(['mouse', 'date', 'probe'], sort=False)])
fraction_by_recording['n_zero_for_ratio'] = fraction_by_recording.n_zero + fraction_by_recording.n_unassessed
fraction_by_recording['on_rf_percent'] = fraction_by_recording.on_rf_percent_all_hd
fraction_by_recording.to_csv(output_dir / 'seven_mice_on_rf_fraction_by_recording.csv', index=False)
coverage = pool_count_coverage(coverage_panels)
assert coverage['n_rf'] == population_summary['n_on_rf_positive']
assert coverage['n_assessed'] == population_summary['n_assessed']
assert {entry['mouse'] for entry in coordinate_records} == {record[0] for record in population_recordings}
np.savez_compressed(output_dir / 'seven_mice_coverage_centered.npz', **coverage)
fig = coverage_figure(coverage)
save_figure(fig, '07_poster_distribution_centered', 'HD + ON RF coverage; same positive cells as fraction, PSTH and direction panels', show=True)

cell_times = pd.DataFrame(cell_time_rows)
cell_times.to_csv(output_dir / 'seven_mice_response_time_by_cell.csv', index=False)
time_columns = sorted(cell_times.time_ms.unique())
cell_matrix = cell_times.pivot(index=['mouse', 'date', 'probe', 'unit_id'], columns='time_ms', values='mean_hz')
# Pool spatial RF bins directly; each measured bin has equal weight.
pooled_moments = response_moments(np.concatenate(pooled_position_rates, axis=0))
timing_mean, timing_sem = response_mean_sem(pooled_moments)
pd.DataFrame(dict(time_ms=time_columns, mean_hz=timing_mean, sem_hz=timing_sem,
                  n_rf_positions=pooled_moments['n'], n_cells=len(cell_matrix))).to_csv(output_dir / 'seven_mice_response_time.csv', index=False)
mouse_times = cell_times.groupby(['mouse', 'time_ms'], sort=False).mean_hz.mean().unstack('time_ms')
mouse_times.to_csv(output_dir / 'seven_mice_response_time_by_mouse.csv')
mouse_mean, mouse_sem = response_mean_sem(response_moments(mouse_times.to_numpy()))
pd.DataFrame(dict(time_ms=time_columns, mean_hz=mouse_mean, sem_hz=mouse_sem,
                  n_mice=len(mouse_times))).to_csv(output_dir / 'seven_mice_response_time_animal_mean.csv', index=False)

peak_tables = []
for mouse, data in direction_tables.items():
    peaks = data['peaks'].copy()
    peaks['mouse'] = mouse
    peak_tables.append(peaks.reset_index())
peak_pairs = pd.concat(peak_tables, ignore_index=True)
assert len(peak_pairs) == coverage['n_rf'] == len(cell_matrix)
peak_pairs.to_csv(output_dir / 'seven_mice_peak_bin_pairs.csv', index=False)
direction_statistics = {mouse: dict(n_cells=int(data['counts'].sum()), counts=data['counts'].tolist(),
    rayleigh_score=float(data['rayleigh_score']), rayleigh_p=float(data['rayleigh_p']))
    for mouse, data in direction_tables.items()}
audit = dict(selection='Saved HD class 3; excitatory ON RF; one paired recording per mouse; both M15 probes',
    anatomy=population_anatomy, fraction_denominator=fraction_denominator, **population_summary,
    all_rf_positive_before_hd_selection=all_rf_count, anatomy_alternatives=anatomy_summaries,
    timing_n_positions=sum(r['n_positions'] for r in timing_records),
    timing_observations='Each saved RF spatial bin contributes equally; descriptive SEM across pooled RF bins, which are correlated within cells',
    direction_statistics=direction_statistics, recordings=verification_records,
    checks=dict(no_duplicate_unit_keys=True, raw_qc_and_detections_reproduced=True,
                identical_positive_cohort_all_panels=True, unassessed_counted_as_zero_for_ratio=True, raw_assessment_status_preserved=True))
(output_dir / 'seven_mice_population_audit.json').write_text(json.dumps(audit, indent=2) + '\n')
with plt.rc_context(poster_style):
    fig = timing_figure(sample_moments, pooled_moments, len(cell_matrix),
                        sum(r['n_positions'] for r in timing_records), unit_id=preview_unit)
    save_figure(fig, '09_poster_sample_and_seven_mice_timing', 'Mean and descriptive SEM across RF bins; population pools all HD+ON RF bins with equal bin weight', show=True)
    fig = fraction_figure(assessment_counts, n_unassessed=population_summary['n_unassessed'], denominator=fraction_denominator)
    save_figure(fig, '10_poster_on_rf_fraction', 'Pie: ON RFs among all HD class-3 cells; unassessed counted as zero by the requested convention', show=True)
fig = seven_mice_figure(direction_tables)
save_figure(fig, '18_poster_seven_mice_peak_sums', 'One all-height RF peak + HD peak per cell, using the common seven-mouse cohort', show=True)
# Separate vector panels let the poster place M22 beside the measured arrows.
for mouse, values in direction_tables.items():
    fig = peak_sum_figure(values, title=mouse.upper())
    save_figure(fig, f'28_poster_{mouse}_peak_sum',
        f'{mouse.upper()}: one all-height RF peak + HD peak per HD class-3 / ON-RF cell')
print(json.dumps(population_summary, indent=2))

# Vector PDF panels and one compact image for reviewing the complete population output.
from matplotlib.backends.backend_pdf import PdfPages
with PdfPages(output_dir / 'seven_mice_reviewed_results.pdf') as pdf:
    figures = [coverage_figure(coverage),
        timing_figure(sample_moments, pooled_moments, len(cell_matrix),
                      sum(r['n_positions'] for r in timing_records), unit_id=preview_unit),
        fraction_figure(assessment_counts, denominator=fraction_denominator),
        seven_mice_figure(direction_tables)]
    for fig in figures:
        pdf.savefig(fig, bbox_inches='tight', facecolor='white', transparent=False)
        plt.close(fig)
fig, axes = plt.subplots(2, 2, figsize=(21, 12), layout='constrained', facecolor='white')
for axis, stem in zip(axes.ravel(), ['07_poster_distribution_centered',
    '09_poster_sample_and_seven_mice_timing', '10_poster_on_rf_fraction', '18_poster_seven_mice_peak_sums']):
    axis.imshow(plt.imread(output_dir / f'{stem}.png'))
    axis.axis('off')
fig.suptitle('Seven mice · HD class 3 · ON excitatory RFs', fontsize=25)
fig.savefig(output_dir / 'seven_mice_reviewed_overview.png', dpi=140, facecolor='white', transparent=False)
plt.close(fig)


## 6. 来源、配对及导出检查

各矩阵的完整 unit key 保存在 CSV 中；poster 上只标 cell rank。RF-HD 的排序来自 S9，HD-EBB 的排序来自 S11。保留原始来源与 SHA-256，所有导出的图为不透明白底。七鼠方向 polar 从当前输入重新计算，沿用每细胞一个 HD + all-height RF peak 的定义，不读取过期静态素材。


In [ ]:
from PIL import Image as PILImage
for entry in figure_records:
    with PILImage.open(output_dir / f'{entry["stem"]}.png') as image:
        rgba = np.asarray(image.convert('RGBA'))
        assert (rgba[..., 3] == 255).all()
        np.testing.assert_array_equal(rgba[0, 0], [255, 255, 255, 255])
metadata = dict(
    mouse='m22', date='261008', probe='A', n_cells=len(unit_ids), sample_unit=int(preview_unit),
    pairing=dict(rf_hd=[2,9], rf_ebb=[2,11], hd_ebb=[11,11], free_moving_rf=10),
    selection='Saved excitatory ON RF in S2 and saved class-3 HD in S9; paired cells retained for wall comparison',
    same_sorting_source=sort_sources[0],
    rotation=dict(units=list(rotation_units), sessions=[2,7], screen_file=str(paths['screen']), animal_file=str(paths['animal']),
                  presentation_exposure='saved stimulusPresentationCounts from each file',
                  normalization='presentation_count_time', window_s=[0,.2], ranges=rotation_ranges,
                  descriptive_map_correlations=rotation_alignment),
    sample_layout=dict(canvas_inches=[18.3,6.25], rectangle_width_inches=8., native_placement_scale=1.,
                       hd_diameter_inches=3., polar_diameter_inches=3.8,
                       colorbars='one horizontal Hz scale below each ON/OFF rectangle'),
    additional_sample=dict(unit=int(multiple_rf_unit), mouse='m22', date='261008', rf_session=2, hd_session=9,
                           n_on_rf=int(detected['ON'].for_unit(multiple_rf_unit, center_only=True).sum()),
                           n_off_rf=int(detected['OFF'].for_unit(multiple_rf_unit, center_only=True).sum()),
                           canvas_inches=[18.3,6.25], rectangle_width_inches=8.,
                           hd_diameter_inches=3., polar_diameter_inches=3.8, native_placement_scale=1.),
    sample_highlight=dict(excluded_zero_based_rc=rf_highlight_exclusions.get(preview_unit, []),
                          saved_on_bins=int(masks[preview_unit, 'ON'].sum()),
                          displayed_on_bins=int(sample_masks['ON'].sum()),
                          affects='sample outlines only; detector masks and analyses unchanged'),
    display=dict(rf_map_format='ON/OFF rectangles plus one ON annulus, stepped frame and mouse icon', fraction_plot='pie with outside 2+ RF callout',
                 hd_rf_panel_height_over_width=hd_rf_panel_aspect, rf_rate_scale='0 to each map maximum Hz', rf_outline='saved contiguous region perimeter',
                 fraction_order=['2+', '1', '0'], free_movement_colorbars='horizontal RF; vertical EBB',
                 free_moving_rectangle_inches=[8., 8. * 84 / 360],
                 rf_colorbar_location='horizontal below lower right; each map uses its own maximum',
                 normalized_heatmap_ticks=['0', 'max'], right_column_shared_colorbars=1,
                 comparison_angle_ticks=[-180,-90,0,90,180], hd_tick_labels=[180,270,0,90,180]),
    rf_window_s=[0,.2], rf_profile_aggregation='all-height raw count sum, peak-normalized after the existing profile preparation',
    elevation_centers_deg=centered_elevation(responses[preview_unit, 'ON'].y_positions).tolist(),
    coverage_elevation_edges_deg=coverage['y_edges'][[0, -1]].tolist(),
    coverage_marginals='Distinct units per azimuth column/elevation row from unsmoothed saved masks',
    coverage_native_grids=coordinate_records, coverage_n_rf=coverage['n_rf'], coverage_n_assessed=coverage['n_assessed'],
    n_mice=population_units.mouse.nunique(), timing_n_units=sum(r['n_units'] for r in timing_records),
    timing_n_positions=sum(r['n_positions'] for r in timing_records),
    timing_sem='Descriptive SEM across pooled RF spatial bins; each bin is equally weighted; bins within cells are correlated; cell-based animal summary exported separately',
    population_anatomy=population_anatomy, population_summary=population_summary,
    anatomy_alternatives=anatomy_summaries, direction_statistics=direction_statistics,
    fraction_counts={key:int(value) for key,value in fraction_counts.items()},
    fraction_rule='All saved HD class-3 cells; no ADN restriction; missing RF records counted as zero at explicit user request; ON excitatory only',
    ebb_session=11, ebb_guides_cm=[8,16], ebb_quality=quality,
    ebb_example=dict(**ebb_example_spec, hd_session=ebb_example_spec['session'], hd_class=3,
                     half_map_r=float(ebb_example_quality['half_r']),
                     selection=joint_metadata['selection'], descriptive_score=float(ebb_example_quality['descriptive_score'])),
    hd_ebb_joint=joint_metadata, poster_hd_ebb_product=slides_product_metadata, rf_region_arrows=region_arrow_statistics,
    rf_ebb_sorting=sorting['metadata'], rf_ebb_pairwise_all=pairwise_metadata['all'],
    rf_ebb_pairwise_bands=pairwise_metadata,
    poster_panel_layout=dict(pairwise='one row: all, 0–8, 8–16, >16 cm',
        direction_rows=[['M22 arrows', 'm22'], ['m14', 'm15', 'm19'], ['m21', 'm23', 'm24']]),
    same_cell_rf_ebb_unit=int(preview_unit),
    free_moving_exposure=dict(method='MATLAB native pixel geometry from stimulus MAT, onsets, XY, HD and calibration',
        screen_width_px=fm_screen_width_px, screen_deg=fm_screen_deg,
        presentation_counts=fm_presentations.tolist(), occupancy_max_abs_error_s=fm_occupancy_max_abs_error),
    ebb_display=dict(source='frame counts and geometry, verified against saved egocentric_rate_maps; display without re-masking',
                     smoothing_sigma_bins=ebb_smoothing_sigma,
                     zero_occupancy_bins=int((occupancy == 0).sum()), finite_display_bins=bool(np.isfinite(ebb_display_rates).all()),
                     saved_smoothing_reconstruction_passed=True,
                     profile_statistics='finite measured-bin means, unchanged'),
    ebb_bands={key:[int(a),int(b)] for key,(a,b) in band_slices(distance_edges).items()},
    sort_orders={key:order.tolist() for key,order in orders.items()},
    figures=figure_records, sources={name:file_record(path) for name,path in source_paths.items()},
    code={str(path):file_record(path) for path in (Path('scripts/poster_figures.py'), Path('scripts/poster_design_figures.py'), Path('scripts/poster_rf_ebb.py'), Path('scripts/rf_ebb_comparison.py'), Path('Utils/tc_comparison.py'),
        Path('scripts/export_m22_rf_sample_variants.py'), Path('scripts/export_m22_f3_examples.py'),
        Path('scripts/slides_v9_figures.py'), Path('scripts/poster_population.py'), Path('EBB/ebc_analysis.py'),
        Path('EBB/quality.py'), Path('EBB/example.py'), Path('EBB/profiles.py'), Path('EBB/joint.py'), Path('EBB/spatial_cell_analysis.py'), Path('RF/free_moving.py'),
        Path('RF/profiles.py'),
        Path('RF/rflocate/rates.py'), Path('RF/rflocate/trials.py'), Path('Utils/plotting.py'))},
    checks=dict(rf_projection_matches_native_counts=rf_projection_max_abs_error == 0, all_pngs_opaque_white=True,
                same_cell_cohort=True, session_pairing_explicit=True, rates_match_counts_over_exposure=True,
                population_common_cohort=True, population_raw_detections_reproduced=True),
)
(output_dir / 'poster_figure_sources.json').write_text(json.dumps(metadata, indent=2)+'\n')
pd.DataFrame(figure_records).to_csv(output_dir / 'figure_index.csv', index=False)
preview_stems = [f'01_rf_polar_unit{preview_unit}_on', f'02_rf_firing_rate_unit{preview_unit}_on', f'03_ebb_polar_unit{ebb_preview_unit}',
    f'04_ebb_polar_8_16cm_unit{ebb_preview_unit}', '13_poster_rf_ebb_sorted', '06_hd_vs_ebb_three_distances_sorted_by_hd']
fig, axes = plt.subplots(3,2,figsize=(18,18),layout='constrained',facecolor='white')
for axis,stem in zip(axes.ravel(),preview_stems):
    axis.imshow(plt.imread(output_dir / f'{stem}.png'))
    axis.axis('off')
fig.savefig(output_dir / 'poster_contact_sheet.png',dpi=130,facecolor='white',transparent=False)
display(fig)
plt.close(fig)
print(f"Verified {len(figure_records)} figures, {len(unit_ids)} paired cells, {coverage['n_rf']} HD + ON RF cells from {population_units.mouse.nunique()} mice.")
